# Aligning MERFISH to H&E staining image from Visium (squidpy + STalign)

In this notebook, we take a single cell resolution spatial transcriptomics dataset of a coronal section of the adult mouse brain profiled by MERFISH and align it to a H&E staining image.

Since this is a point-to-image alignment, we use the STalign/LDDMM backend via `squidpy`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import anndata as ad
import squidpy as sq

plt.rcParams["figure.figsize"] = (12, 10)

## Load MERFISH cell data (source)

In [ ]:
fname = '../merfish_data/datasets_mouse_brain_map_BrainReceptorShowcase_Slice2_Replicate3_cell_metadata_S2R3.csv.gz'
df1 = pd.read_csv(fname)

coords_source = np.column_stack([df1['center_x'].values, df1['center_y'].values])
adata_source = ad.AnnData(X=np.zeros((len(coords_source), 1)), obs=df1)
adata_source.obsm['spatial'] = coords_source

fig, ax = plt.subplots()
ax.scatter(coords_source[:, 0], coords_source[:, 1], s=1, alpha=0.2)
ax.set_title('MERFISH cell positions (source)')

## Load H&E image (target)

In [ ]:
image_file = '../visium_data/tissue_hires_image.png'
V = plt.imread(image_file)

fig, ax = plt.subplots()
ax.imshow(V)
ax.set_title('H&E staining image (target)')
print(f"Image shape: {V.shape}")

## Load landmark points

For point-to-image alignment, landmark points help initialize the alignment.

In [ ]:
# Load pre-computed landmark points
data = np.load('../visium_data/visium2_points.npz')
pointsI = np.array(data['pointsI'][..., ::-1])  # to x,y
pointsJ = np.array(data['pointsJ'][..., ::-1])  # to x,y
print(f"Source landmarks: {pointsI}")
print(f"Target landmarks: {pointsJ}")

## Align using STalign (point-to-image)

`sq.experimental.tl.align` automatically selects the STalign backend when aligning points to an image.

In [ ]:
# Align MERFISH cell positions to H&E image
# landmark_source/landmark_target are in (row, col) i.e. (y, x) order in the original STalign convention
# but sq.experimental.tl.align expects (x, y) order
sq.experimental.tl.align(
    adata_source,
    V,  # target image
    method='lddmm',
    resolution=30.0,
    niter=200,
    diffeo_start=100,
    landmark_source=pointsI,  # (y, x) landmarks from original format
    landmark_target=pointsJ,
    verbose=True,
    sigmaM=0.2,
    sigmaB=0.19,
    sigmaA=0.3,
    sigmaP=2e-1,
    epL=5e-11,
    epT=5e-4,
    epV=5e1,
)

## Visualize results

In [ ]:
aligned = adata_source.obsm['spatial_aligned']
extentJ = [0, V.shape[1], 0, V.shape[0]]

fig, ax = plt.subplots()
ax.imshow(V, extent=extentJ)
ax.scatter(aligned[:, 0], aligned[:, 1], s=1, alpha=0.1, label='MERFISH aligned')
ax.set_title('Aligned MERFISH on H&E')
ax.legend(markerscale=10)

## Save results

In [ ]:
df_aligned = pd.DataFrame({'aligned_x': aligned[:, 0], 'aligned_y': aligned[:, 1]})
results = pd.concat([df1, df_aligned], axis=1)
results.head()